In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [ ]:
samples_pb = pd.read_csv('../hapmap/pacbio/samples.csv', names=['sample', 'file'])['sample'].to_list()
samples_ont = pd.read_csv('../hapmap/ont/samples.csv', names=['sample', 'file'])['sample'].to_list()

In [ ]:
def calculate_N50(list_of_lengths):
    # Sort the lengths in descending order
    sorted_lengths = sorted(list_of_lengths, reverse=True)
    
    # Calculate half of the total sum of lengths
    total_length = sum(sorted_lengths)
    half_total = total_length / 2
    
    # Iterate through the sorted lengths to find N50
    cumulative_sum = 0
    n50_value = 0
    for length in sorted_lengths:
        cumulative_sum += length
        if cumulative_sum >= half_total:
            n50_value = length
            break # Stop when the condition is met
            
    return n50_value

In [ ]:
dict_of_length_lists = {}
df_lr = pd.DataFrame(columns=['Sample', 'Total_Mapped_Reads', 'Median_Read_Length', 'N50'])

for sample in samples_pb + samples_ont:
    file_path=f"/net/nwgc/vol1/home/czaka/analysis/mitoscope/smaht/analysis/read_lengths_full_bams/{sample}.read_lengths.txt"
    with open(file_path, 'r') as fr:
        list_of_read_lengths = [int(line.strip()) for line in fr] 
        dict_of_length_lists[sample] = list_of_read_lengths
        num_reads = len(list_of_read_lengths)
        med_length = np.median(list_of_read_lengths)
        n50 = calculate_N50(dict_of_length_lists[sample])
        df_lr.loc[len(df_lr)] = [sample, num_reads, med_length, n50]

In [ ]:
df_lr.to_csv('tables/hapmap_LR_whole_genome_stats.csv', index=False)

In [ ]:
dict_of_length_lists_sr = {}
df_sr = pd.DataFrame(columns=['Sample', 'Total_Mapped_Read_Pairs', 'Mito_Coverage', 'Nuclear_Coverage', 'mtDNA_CN'])

samples_illumina = pd.read_csv('../../../mutect2/smaht/illumina/hapmap/samples.csv')['sample_id'].to_list()
for sample in samples_illumina:
    # total mapped read pairs
    file_path=f"./short_read_total_counts/{sample}.total_read_count.txt"
    with open(file_path, 'r') as fr:
        total_read_pairs = fr.readline().strip()

    # mito coverage
    mt_cov_file_path=f"./short_read_coverage/chrM/{sample}.mosdepth.summary.txt"
    mito_cov_df = pd.read_csv(mt_cov_file_path, sep='\t')
    mito_cov = mito_cov_df[mito_cov_df ['chrom'] == 'chrM']['mean'].iloc[0]

    # mito coverage
    nuc_cov_file_path=f"./short_read_coverage/{sample}.mosdepth.summary.txt"
    nuc_cov_df = pd.read_csv(nuc_cov_file_path, sep='\t')
    nuc_cov = nuc_cov_df[nuc_cov_df ['chrom'] == 'total']['mean'].iloc[0]

    mtdna_cn = (mito_cov / nuc_cov) * 2


    df_sr.loc[len(df_sr)] = [sample, total_read_pairs, mito_cov, nuc_cov, mtdna_cn]

df_sr

df_sr.to_csv('tables/hapmap_SR_whole_genome_stats.csv')

In [ ]:
df_pb = pd.read_csv("../hapmap/pacbio/output/qc_summary.tsv", sep='\t')
df_ont = pd.read_csv("../hapmap/ont/output/qc_summary.tsv", sep='\t')

df = pd.concat([df_pb, df_ont]).reset_index()

df_w_CN = pd.merge(df_lr, df, on='Sample')
df_w_CN[['Donor', 'Tech', 'Center']] = df_w_CN['Sample'].str.split('-', expand=True)

In [ ]:
sns.set_theme(style="ticks", context="talk", font_scale=0.8)

common_kws = dict(scatter_kws={'s': 40, 'alpha': 0.7, 'edgecolor': 'black'},
                  line_kws={'color': 'red', 'lw': 2})

sns.regplot(df_w_CN, x='mtDNA_CN', y='Median_Read_Length_x', ci=None, **common_kws)
plt.xlabel('Mitochondrial Copy Number')
plt.ylabel('Median Read Length')
#plt.savefig(f"plots/mtdna_cn_vs_med_read_length.pdf", dpi=300, bbox_inches='tight')
plt.show()